In [1]:
# box,pie,diandian

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import joblib
import os
# from outlier_cleaner import OutlierCleaner

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder,TargetEncoder,LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor


In [3]:
le = LabelEncoder()

In [4]:
df = pd.read_csv('data_science_salaries.csv')

In [5]:
df.head()

,job_title,experience_level,employment_type,work_models,work_year,employee_residence,salary,salary_currency,salary_in_usd,company_location,company_size
0,Data Engineer,Mid-level,Full-time,Remote,2024,United States,148100,USD,148100,United States,Medium
1,Data Engineer,Mid-level,Full-time,Remote,2024,United States,98700,USD,98700,United States,Medium
2,Data Scientist,Senior-level,Full-time,Remote,2024,United States,140032,USD,140032,United States,Medium
3,Data Scientist,Senior-level,Full-time,Remote,2024,United States,100022,USD,100022,United States,Medium
4,BI Developer,Mid-level,Full-time,On-site,2024,United States,120000,USD,120000,United States,Medium


In [6]:
df.columns

Index(['job_title', 'experience_level', 'employment_type', 'work_models',
       'work_year', 'employee_residence', 'salary', 'salary_currency',
       'salary_in_usd', 'company_location', 'company_size'],
      dtype='object')

In [7]:
df.shape

(6599, 11)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6599 entries, 0 to 6598
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   job_title           6599 non-null   object
 1   experience_level    6599 non-null   object
 2   employment_type     6599 non-null   object
 3   work_models         6599 non-null   object
 4   work_year           6599 non-null   int64 
 5   employee_residence  6599 non-null   object
 6   salary              6599 non-null   int64 
 7   salary_currency     6599 non-null   object
 8   salary_in_usd       6599 non-null   int64 
 9   company_location    6599 non-null   object
 10  company_size        6599 non-null   object
dtypes: int64(3), object(8)
memory usage: 567.2+ KB


In [9]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
6594    False
6595    False
6596    False
6597    False
6598    False
Length: 6599, dtype: bool

In [10]:
df.isnull().sum()

job_title             0
experience_level      0
employment_type       0
work_models           0
work_year             0
employee_residence    0
salary                0
salary_currency       0
salary_in_usd         0
company_location      0
company_size          0
dtype: int64

# Encoding

In [11]:
df.head()

,job_title,experience_level,employment_type,work_models,work_year,employee_residence,salary,salary_currency,salary_in_usd,company_location,company_size
0,Data Engineer,Mid-level,Full-time,Remote,2024,United States,148100,USD,148100,United States,Medium
1,Data Engineer,Mid-level,Full-time,Remote,2024,United States,98700,USD,98700,United States,Medium
2,Data Scientist,Senior-level,Full-time,Remote,2024,United States,140032,USD,140032,United States,Medium
3,Data Scientist,Senior-level,Full-time,Remote,2024,United States,100022,USD,100022,United States,Medium
4,BI Developer,Mid-level,Full-time,On-site,2024,United States,120000,USD,120000,United States,Medium


In [12]:
Target = "salary_in_usd"

In [13]:
df[Target].describe()

count      6599.000000
mean     145560.558569
std       70946.838070
min       15000.000000
25%       95000.000000
50%      138666.000000
75%      185000.000000
max      750000.000000
Name: salary_in_usd, dtype: float64

## Ordinal encoder

In [14]:
df.experience_level.unique()

array(['Mid-level', 'Senior-level', 'Entry-level', 'Executive-level'],
      dtype=object)

In [15]:
experience_order = [[ 'Executive-level', 'Entry-level', 'Senior-level','Mid-level']]

In [16]:
ord_encoder = OrdinalEncoder()
df['experience_encoded'] = ord_encoder.fit_transform(df[['experience_level']])

for i ,cat in enumerate(experience_order[0]):
    print(f"{cat}-{i}")

Executive-level-0
Entry-level-1
Senior-level-2
Mid-level-3


In [17]:
df.company_size.unique()

array(['Medium', 'Large', 'Small'], dtype=object)

In [18]:
company_order = [[ 'Small', 'Large','Medium']]

In [19]:
ord_encoder = OrdinalEncoder()

df['company_encoded'] = ord_encoder.fit_transform(df[['company_size']])

for i ,cat in enumerate(company_order[0]):
    print(f"{cat}-{i}")

Small-0
Large-1
Medium-2


## Target Encoder

In [20]:
# ! pip install category_encoders

# -------------------------------

## 1.Job

In [21]:
df.job_title.unique()

array(['Data Engineer', 'Data Scientist', 'BI Developer',
       'Research Analyst', 'Business Intelligence Developer',
       'Data Analyst', 'Director of Data Science', 'MLOps Engineer',
       'Machine Learning Scientist', 'Machine Learning Engineer',
       'Data Science Manager', 'Applied Scientist',
       'Business Intelligence Analyst', 'Analytics Engineer',
       'Business Intelligence Engineer', 'Data Science',
       'Research Scientist', 'Research Engineer',
       'Managing Director Data Science', 'AI Engineer', 'Data Specialist',
       'Data Architect', 'Data Visualization Specialist', 'ETL Developer',
       'Data Science Practitioner', 'Computer Vision Engineer',
       'Data Lead', 'ML Engineer', 'Data Developer', 'Data Modeler',
       'Data Science Consultant', 'AI Architect',
       'Data Analytics Manager', 'Data Science Engineer',
       'Data Product Manager', 'Data Quality Analyst', 'Data Strategist',
       'Prompt Engineer', 'Data Science Lead',
       'Busi

In [22]:
# AI

def map_job_category(title: str) -> str:
    title = title.lower()

    if any(k in title for k in [
        'data engineer', 'etl', 'big data', 'cloud data', 'azure data',
        'aws data', 'data infrastructure', 'data integration',
        'data devops', 'pipeline', 'database'
    ]):
        return 'Data Engineer'

    elif any(k in title for k in [
        'machine learning', 'ml engineer', 'mlops',
        'ai engineer', 'ai developer', 'deep learning',
        'nlp', 'computer vision'
    ]):
        return 'Machine Learning / AI'

    elif any(k in title for k in [
        'data scientist', 'data science', 'applied scientist',
        'research scientist', 'decision scientist'
    ]):
        return 'Data Scientist'

    elif any(k in title for k in [
        'business intelligence', 'bi developer', 'bi analyst', 'power bi'
    ]):
        return 'Business Intelligence'

    elif any(k in title for k in [
        'data analyst', 'analytics', 'insight analyst'
    ]):
        return 'Data Analyst'

    elif any(k in title for k in [
        'architect', 'data model', 'modeller', 'modeler', 'visualization'
    ]):
        return 'Architecture / Modeling'

    elif any(k in title for k in [
        'manager', 'director', 'head', 'lead', 'principal'
    ]):
        return 'Management'

    elif any(k in title for k in [
        'quality', 'governance', 'strategy', 'operations', 'compliance'
    ]):
        return 'Data Management / Quality'

    else:
        return 'Other'


In [23]:
job_category = [[ 'Other', 'Management','Architecture / Modeling','Data Analyst','Business Intelligence','Data Scientist','Machine Learning / AI','Data Engineer']]

In [24]:
df['job_category'] = df['job_title'].apply(map_job_category)

df[['job_title', 'job_category']].head()


,job_title,job_category
0,Data Engineer,Data Engineer
1,Data Engineer,Data Engineer
2,Data Scientist,Data Scientist
3,Data Scientist,Data Scientist
4,BI Developer,Business Intelligence


In [25]:
df['job_category'].unique()

array(['Data Engineer', 'Data Scientist', 'Business Intelligence',
       'Other', 'Data Analyst', 'Machine Learning / AI',
       'Architecture / Modeling', 'Management',
       'Data Management / Quality'], dtype=object)

In [26]:
df[['job_title']].value_counts().values

array([1307, 1243,  910,  629,  246,  206,  176,  136,  113,   97,   86,
         82,   73,   69,   63,   60,   56,   48,   47,   42,   41,   31,
         31,   31,   29,   28,   28,   24,   24,   22,   22,   21,   18,
         18,   17,   17,   16,   16,   16,   15,   15,   15,   15,   14,
         14,   13,   12,   12,   12,   12,   12,   12,   10,   10,   10,
         10,   10,   10,    8,    8,    8,    6,    6,    6,    6,    6,
          6,    5,    5,    5,    5,    5,    5,    4,    4,    4,    4,
          4,    4,    4,    4,    4,    4,    4,    4,    4,    3,    3,
          3,    3,    3,    3,    3,    3,    3,    2,    2,    2,    2,
          2,    2,    2,    2,    2,    2,    2,    2,    2,    2,    2,
          2,    2,    2,    2,    2,    2,    1,    1,    1,    1,    1,
          1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1],
      dtype=int64)

In [27]:
label_encoder = LabelEncoder()

df['job_cat_encoded']= label_encoder.fit_transform(df['job_category'])
# a = target_encoder.fit_transform(df['job_title'],df['salary_in_usd'])

# for cat, val in job_te_map.items():
#     print(f"{cat} -> {val:.2f}")

df['job_clean'] = df['job_cat_encoded']
job_clean = df['job_clean']

In [28]:
label_encoder.inverse_transform(job_clean)


array(['Data Engineer', 'Data Engineer', 'Data Scientist', ...,
       'Machine Learning / AI', 'Data Engineer', 'Data Scientist'],
      dtype=object)

In [29]:
df.columns

Index(['job_title', 'experience_level', 'employment_type', 'work_models',
       'work_year', 'employee_residence', 'salary', 'salary_currency',
       'salary_in_usd', 'company_location', 'company_size',
       'experience_encoded', 'company_encoded', 'job_category',
       'job_cat_encoded', 'job_clean'],
      dtype='object')

# -------------------------------

## Employee residence

In [30]:
df.employee_residence.unique()

array(['United States', 'Romania', 'Portugal', 'Lebanon', 'India',
       'Ireland', 'United Kingdom', 'Spain', 'Egypt', 'Germany', 'Canada',
       'Australia', 'Vietnam', 'Uganda', 'Italy', 'Thailand',
       'Philippines', 'Tunisia', 'Belgium', 'Turkey', 'Nigeria',
       'Uzbekistan', 'Argentina', 'France', 'Kuwait', 'Ukraine',
       'South Africa', 'Slovenia', 'Singapore', 'Sweden', 'Saudi Arabia',
       'Armenia', 'Russia', 'Georgia', 'Qatar', 'Peru', 'Poland',
       'New Zealand', 'Norway', 'Netherlands', 'Mexico', 'Latvia',
       'Pakistan', 'Lithuania', 'South Korea', 'Kenya', 'Moldova',
       'Israel', 'Hungary', 'Croatia', 'Hong Kong', 'Greece', 'Ghana',
       'Finland', 'Estonia', 'Cyprus', 'Ecuador', 'Austria', 'Colombia',
       'Switzerland', 'Central African Republic', 'Czech Republic',
       'Brazil', 'Bosnia and Herzegovina', 'Andorra', 'China',
       'Costa Rica', 'Chile', 'Bolivia', 'Malaysia',
       'United Arab Emirates', 'Puerto Rico', 'Japan', 'Iran',
 

In [31]:
## Ai

In [32]:
def map_country_region(country):
    if country in ['United States', 'Canada', 'Mexico', 'Puerto Rico']:
        return 'North America'

    elif country in [
        'United Kingdom', 'Ireland', 'Germany', 'France', 'Netherlands',
        'Belgium', 'Switzerland', 'Austria', 'Sweden', 'Norway',
        'Denmark', 'Finland', 'Italy', 'Spain', 'Portugal',
        'Greece', 'Luxembourg', 'Malta', 'Cyprus', 'Andorra', 'Jersey'
    ]:
        return 'Western Europe'

    elif country in [
        'Poland', 'Romania', 'Ukraine', 'Czech Republic', 'Hungary',
        'Croatia', 'Slovenia', 'Serbia', 'Bosnia and Herzegovina',
        'Bulgaria', 'Lithuania', 'Latvia', 'Estonia', 'Moldova',
        'Armenia', 'Georgia', 'Russia'
    ]:
        return 'Eastern Europe'

    elif country in ['Singapore', 'Japan', 'South Korea', 'Hong Kong']:
        return 'Asia Developed'

    elif country in [
        'India', 'China', 'Malaysia', 'Thailand', 'Vietnam',
        'Philippines', 'Indonesia', 'Pakistan', 'Iran',
        'Iraq', 'Uzbekistan'
    ]:
        return 'Asia Developing'

    elif country in [
        'United Arab Emirates', 'Saudi Arabia', 'Qatar',
        'Kuwait', 'Israel', 'Lebanon', 'Turkey'
    ]:
        return 'Middle East'

    elif country in [
        'South Africa', 'Egypt', 'Nigeria', 'Kenya', 'Ghana',
        'Uganda', 'Tunisia', 'Algeria', 'Mauritius',
        'Central African Republic'
    ]:
        return 'Africa'

    elif country in [
        'Brazil', 'Argentina', 'Chile', 'Colombia',
        'Peru', 'Bolivia', 'Ecuador'
    ]:
        return 'South America'

    elif country in ['Australia', 'New Zealand', 'American Samoa']:
        return 'Oceania'

    else:
        return 'Other'


In [33]:
residence_category = [[ 'Other', 'Oceania','South America','Africa','Middle East','Asia Developing','Asia Developed','Eastern Europe','Western Europe','North America']]

In [34]:
df['residence_category'] = df['employee_residence'].apply(map_country_region)
df['residence_category'].value_counts()

residence_category
North America      5561
Western Europe      715
Asia Developing     100
Eastern Europe       71
South America        48
Africa               34
Oceania              33
Middle East          17
Asia Developed       17
Other                 3
Name: count, dtype: int64

In [35]:
df['residence_category'].unique()

array(['North America', 'Eastern Europe', 'Western Europe', 'Middle East',
       'Asia Developing', 'Africa', 'Oceania', 'South America',
       'Asia Developed', 'Other'], dtype=object)

In [36]:
df[['residence_category']].value_counts().values

array([5561,  715,  100,   71,   48,   34,   33,   17,   17,    3],
      dtype=int64)

In [37]:
label_encoder = LabelEncoder()

df['region_encoded']= label_encoder.fit_transform(df['residence_category'])

df.region_encoded

0       5
1       5
2       5
3       5
4       5
       ..
6594    5
6595    0
6596    5
6597    9
6598    9
Name: region_encoded, Length: 6599, dtype: int32

In [38]:
label_encoder.inverse_transform

<bound method LabelEncoder.inverse_transform of LabelEncoder()>

# --------------------------------------------------------

In [39]:
df.company_location.unique()

array(['United States', 'Romania', 'Portugal', 'Lebanon', 'India',
       'Ireland', 'United Kingdom', 'Spain', 'Egypt', 'Germany', 'Canada',
       'Australia', 'Vietnam', 'Ukraine', 'Turkey', 'Thailand',
       'South Africa', 'Slovenia', 'Singapore', 'Sweden', 'Saudi Arabia',
       'Russia', 'Qatar', 'Poland', 'Philippines', 'New Zealand',
       'Norway', 'Netherlands', 'Nigeria', 'Mexico', 'Latvia',
       'Luxembourg', 'Lithuania', 'South Korea', 'Kenya', 'Japan',
       'Italy', 'Israel', 'Hungary', 'Croatia', 'Hong Kong', 'Greece',
       'Gibraltar', 'Ghana', 'France', 'Finland', 'Estonia', 'Ecuador',
       'Denmark', 'Czech Republic', 'Colombia', 'Switzerland',
       'Central African Republic', 'Brazil', 'Bosnia and Herzegovina',
       'Armenia', 'Argentina', 'Andorra', 'United Arab Emirates',
       'Puerto Rico', 'Pakistan', 'Malaysia', 'Iran', 'Indonesia',
       'Algeria', 'Belgium', 'Austria', 'Malta', 'Moldova', 'Iraq',
       'China', 'Chile', 'Bahamas', 'Mauritius

### Ai

In [40]:
def map_country_region(country):
    if country in [
        'United States', 'Canada', 'Mexico', 'Puerto Rico', 'Bahamas'
    ]:
        return 'North America'

    elif country in [
        'United Kingdom', 'Ireland', 'Germany', 'France', 'Netherlands',
        'Belgium', 'Switzerland', 'Austria', 'Sweden', 'Norway',
        'Denmark', 'Finland', 'Italy', 'Spain', 'Portugal',
        'Greece', 'Luxembourg', 'Malta', 'Andorra', 'Gibraltar'
    ]:
        return 'Western Europe'

    elif country in [
        'Poland', 'Romania', 'Ukraine', 'Czech Republic', 'Hungary',
        'Croatia', 'Slovenia', 'Bosnia and Herzegovina',
        'Lithuania', 'Latvia', 'Estonia', 'Moldova',
        'Armenia', 'Russia'
    ]:
        return 'Eastern Europe'

    elif country in [
        'Singapore', 'Japan', 'South Korea', 'Hong Kong', 'Israel'
    ]:
        return 'Asia Developed'

    elif country in [
        'India', 'China', 'Malaysia', 'Thailand', 'Vietnam',
        'Philippines', 'Indonesia', 'Pakistan', 'Iran', 'Iraq'
    ]:
        return 'Asia Developing'

    elif country in [
        'United Arab Emirates', 'Saudi Arabia', 'Qatar', 'Turkey'
    ]:
        return 'Middle East'

    elif country in [
        'South Africa', 'Egypt', 'Nigeria', 'Kenya', 'Ghana',
        'Algeria', 'Mauritius', 'Central African Republic'
    ]:
        return 'Africa'

    elif country in [
        'Brazil', 'Argentina', 'Chile', 'Colombia', 'Ecuador'
    ]:
        return 'South America'

    elif country in [
        'Australia', 'New Zealand'
    ]:
        return 'Oceania'

    else:
        return 'Other'


In [41]:
map_category = [[ 'Other','Oceania,South','America,Africa','Middle East','Asia Developing','Asia Developed','Eastern Europe','Western Europe','North America']]

In [42]:
df['map_category'] = df['company_location'].apply(map_country_region)
df['map_category'].value_counts()


map_category
North America      5613
Western Europe      714
Asia Developing      76
Eastern Europe       64
South America        38
Oceania              35
Africa               26
Asia Developed       19
Middle East          11
Other                 3
Name: count, dtype: int64

In [43]:
df['map_category'].unique()

array(['North America', 'Eastern Europe', 'Western Europe', 'Other',
       'Asia Developing', 'Africa', 'Oceania', 'Middle East',
       'Asia Developed', 'South America'], dtype=object)

In [44]:
df[['map_category']].value_counts().values

array([5613,  714,   76,   64,   38,   35,   26,   19,   11,    3],
      dtype=int64)

In [45]:
label_encoder = LabelEncoder()

df['map_encoded']= label_encoder.fit_transform(df['map_category'])
df.map_encoded

0       5
1       5
2       5
3       5
4       5
       ..
6594    5
6595    5
6596    5
6597    9
6598    9
Name: map_encoded, Length: 6599, dtype: int32

In [46]:
# label_encoder.inverse_transform(a)

In [47]:
df.to_csv('cleaned_salary.csv')

## -----------------------------------------------------------------------------------

# Modeling

In [48]:
USE_COLS = [
    # UI / 业务输入（文字）
    "job_category",
    "experience_level",
    "employment_type",
    "work_models",
    "employee_residence",
    "company_location",
    "company_size",

    # 数值
    "work_year",

    # target
    "salary_in_usd"
]
df = pd.read_csv("cleaned_salary.csv", usecols=USE_COLS)

In [49]:
df.columns

Index(['experience_level', 'employment_type', 'work_models', 'work_year',
       'employee_residence', 'salary_in_usd', 'company_location',
       'company_size', 'job_category'],
      dtype='object')

In [50]:
X = df.drop(columns=["salary_in_usd"])
y = df["salary_in_usd"]


In [51]:
num_cols = [
    "work_year",
    "experience_encoded",
    "company_encoded",
    "job_cat_encoded",
    "region_encoded",
    "map_encoded"
]
cat_cols = [
    "job_category",
    "experience_level",
    "employment_type",
    "work_models",
    "employee_residence",
    "company_location",
    "company_size"
]


In [52]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)


In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(5279, 8) (1320, 8)


In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [55]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42,
    n_estimators=200
)


In [56]:
num_cols = X.select_dtypes(include="number").columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

pipe.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse ma

## Prediction Function

In [57]:
import pandas as pd
import joblib
joblib.dump(pipe, "salary_pipeline.joblib")
pipe = joblib.load("salary_pipeline.joblib")

def predict_salary(input_dict):
    input_df = pd.DataFrame([input_dict])
    pred = pipe.predict(input_df)
    return round(float(pred[0]), 2)


In [58]:
df.columns

Index(['experience_level', 'employment_type', 'work_models', 'work_year',
       'employee_residence', 'salary_in_usd', 'company_location',
       'company_size', 'job_category'],
      dtype='object')

In [59]:
def predict_ppg(work_year, salary,job_cat_encoded,
       experience_level_encoded, employment_type_encoded,
       work_models_encoded, employee_residence_encoded,
       company_location_encoded, company_size_encoded,
       salary_currency_encoded):
    model = joblib.load('salary.joblib')
    input_arr = np.array([[ 
        work_year, salary,job_cat_encoded,
       experience_level_encoded, employment_type_encoded,
       work_models_encoded, employee_residence_encoded,
       company_location_encoded, company_size_encoded,
       salary_currency_encoded
    ]])
    predict = model.predict(input_arr)
    return predict.round(2)[0]

In [60]:
df.columns

Index(['experience_level', 'employment_type', 'work_models', 'work_year',
       'employee_residence', 'salary_in_usd', 'company_location',
       'company_size', 'job_category'],
      dtype='object')

## _________________________________________________________

# App( 100%AI )

In [61]:
import gradio as gr
import pandas as pd
import joblib

# ======================
# Load pipeline & data
# ======================
pipe = joblib.load("salary_pipeline.joblib")
df = pd.read_csv("cleaned_salary.csv")

# ======================
# UI choices (TEXT, unchanged)
# ======================
job_title_choices = sorted(df["job_category"].dropna().astype(str).unique().tolist())
experience_choices = sorted(df["experience_level"].dropna().astype(str).unique().tolist())
employment_choices = sorted(df["employment_type"].dropna().astype(str).unique().tolist())
work_models_choices = sorted(df["work_models"].dropna().astype(str).unique().tolist())
residence_choices = sorted(df["employee_residence"].dropna().astype(str).unique().tolist())
location_choices = sorted(df["company_location"].dropna().astype(str).unique().tolist())
company_size_choices = sorted(df["company_size"].dropna().astype(str).unique().tolist())

year_min = int(df["work_year"].min())
year_max = int(df["work_year"].max())

# ======================
# Predict function
# ======================
def predict_salary(
    job_title,
    experience_level,
    employment_type,
    work_models,
    work_year,
    employee_residence,
    company_location,
    company_size
):
    row = {
        "job_category": job_title,
        "experience_level": experience_level,
        "employment_type": employment_type,
        "work_models": work_models,
        "work_year": int(work_year),
        "employee_residence": employee_residence,
        "company_location": company_location,
        "company_size": company_size,
    }

    X = pd.DataFrame([row])
    pred = pipe.predict(X)[0]
    return round(float(pred), 2)

# ======================
# Gradio App
# ======================
with gr.Blocks(title="Salary Prediction Demo") as demo:
    gr.Markdown("# Salary Prediction Demo")

    with gr.Row():
        job_title_ui = gr.Dropdown(job_title_choices, label="Job Title")
        experience_ui = gr.Dropdown(experience_choices, label="Experience Level")
        employment_ui = gr.Dropdown(employment_choices, label="Employment Type")
        work_model_ui = gr.Dropdown(work_models_choices, label="Work Model")

    with gr.Row():
        year_ui = gr.Slider(year_min, year_max, step=1, value=year_max, label="Work Year")
        residence_ui = gr.Dropdown(residence_choices, label="Employee Residence")
        location_ui = gr.Dropdown(location_choices, label="Company Location")
        company_size_ui = gr.Dropdown(company_size_choices, label="Company Size")

    predict_btn = gr.Button("Predict")
    output = gr.Number(label="Predicted Salary (USD)")

    predict_btn.click(
        fn=predict_salary,
        inputs=[
            job_title_ui,
            experience_ui,
            employment_ui,
            work_model_ui,
            year_ui,
            residence_ui,
            location_ui,
            company_size_ui
        ],
        outputs=output
    )


demo.launch()



* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [62]:
df.columns

Index(['Unnamed: 0', 'job_title', 'experience_level', 'employment_type',
       'work_models', 'work_year', 'employee_residence', 'salary',
       'salary_currency', 'salary_in_usd', 'company_location', 'company_size',
       'experience_encoded', 'company_encoded', 'job_category',
       'job_cat_encoded', 'job_clean', 'residence_category', 'region_encoded',
       'map_category', 'map_encoded'],
      dtype='object')